# H.4.1 scraping -- clean rebuild

Replaces `ScrapingH41Tables.ipynb`. Scope: HTML-format releases from
**1997-03-06 onward** only (ALFRED covers aggregates from ~2002-12 but has
no maturity-bucket detail; pre-1997 data comes from a different source, not
this notebook).

Logic lives in `Helpers/h41_scrape_helpers.py`:
- candidate dates = every Thursday in range (+/- a few days for holiday
  shifts), not every calendar day
- idempotent: already-downloaded files are never re-fetched
- resumable: a manifest.csv tracks status per date across runs
- a `max_requests` cap lets you sanity-check a small slice before
  committing to the full historical run

In [ ]:
import os, sys
from datetime import datetime

base_dir = os.path.dirname(os.getcwd())  # .../PAPER 1 - STRUCTURAL BREAKS
sys.path.append(os.path.join(base_dir, "Helpers"))
from h41_scrape_helpers import run_scrape, summarize_manifest, FIRST_RELEASE

OUT_DIR = os.path.join(base_dir, "Data Input", "Raw Input", "h41_html")
print("Output dir:", OUT_DIR)
print("First release date used:", FIRST_RELEASE.date())

## Step 1 -- small test run

Fetches at most 5 NEW pages (cached hits don't count against the cap) to
confirm the request/parsing/save logic actually works before committing to
a multi-hundred-request historical run.

In [ ]:
test_manifest = run_scrape(
    out_dir=OUT_DIR,
    start=FIRST_RELEASE,
    end=datetime(1997, 6, 1),
    max_requests=5,
)
test_manifest

## Step 2 -- full historical run

This will make on the order of a few hundred live requests to
federalreserve.gov (one per week since 1997, minus whatever Step 1 already
cached). At `sleep_sec=1.0` that's roughly 10-15 minutes, not hours, thanks
to only checking Thursdays instead of every day.

**Left commented out deliberately -- uncomment and run only when you're
ready to actually kick off the full scrape.**

In [ ]:
# full_manifest = run_scrape(
#     out_dir=OUT_DIR,
#     start=FIRST_RELEASE,
#     end=datetime.today(),
#     sleep_sec=1.0,
# )
# summarize_manifest(full_manifest)

## Step 3 -- check results

Re-running `run_scrape` any time (e.g. weekly, to pick up new releases) only
fetches dates not already resolved in the manifest.

In [ ]:
manifest_path = os.path.join(OUT_DIR, "manifest.csv")
if os.path.exists(manifest_path):
    import pandas as pd
    m = pd.read_csv(manifest_path, parse_dates=["date"])
    print(summarize_manifest(m))
    display(m[m["status"] == "unexpected_content"])  # worth a manual look if any
else:
    print("No manifest yet -- run Step 1 or Step 2 first.")

## Step 4 -- incremental update (run this periodically going forward)

Once the full historical backfill (Step 2) has completed, use this instead
of re-running the full 1997-today scan. It finds the most recent release
already on disk and only checks from a short buffer before it through
today -- the thing to re-run weekly (or whenever) to pick up newly
published releases.

Do not run this while Step 2's full run is still in progress -- both write
to the same manifest.csv and a concurrent run can clobber the other's
progress.

In [ ]:
from h41_scrape_helpers import update_h41_archive

# update_manifest = update_h41_archive(out_dir=OUT_DIR)
# summarize_manifest(update_manifest)